# 06 · PD lifetime por rating (só PD)

**Etapa 8 do pipeline.** Estende a PD de 12 meses de cada rating para a vida do contrato, pela **costura de PDs condicionais por idade** (o método de safra, *vintage*).

| passo | onde roda | o que faz |
|---|---|---|
| 1 | Spark → tabela pequena | agrega duração e evento por rating, safra e partição (uma passada) |
| 2 | pandas | hazard por rating e idade, com o **DES** (treino + teste) |
| 3 | pandas | extrapola as idades com poucos contratos em risco |
| 4 | pandas | calibra cada curva para a PD de 12 meses bater a PD da régua |
| 5 | pandas | backtest: Kaplan-Meier no **OOT** e no **HIST**, com intervalo de Greenwood |
| 6 | Spark | aplica a curva aos contratos vivos na data de corte |
| 7 | pandas | gráficos 8.1 a 8.5 |

## Conceitos

**Hazard (PD condicional)** na idade t: dos contratos que chegaram vivos e observados ao mês t, a fração que entrou em default exatamente nesse mês.

```
hazard(t) = defaults na idade t / contratos em risco na idade t
```

**Em risco na idade t** = contratos acompanhados até pelo menos a idade t. Um contrato **censurado** (quitado ou ainda vivo na data de corte) conta como em risco até o mês em que saiu da observação, e depois deixa a conta. Tratá-lo como "bom para sempre" subestimaria o risco das idades altas.

**Costura:**

```
sobrevivência  S(t) = (1 − h₁)·(1 − h₂)·…·(1 − hₜ)
PD acumulada   F(t) = 1 − S(t)
PD marginal    m(t) = S(t−1)·h(t)     (chance de quebrar exatamente no mês t, vista de hoje)
```

**Exemplo (rating H):** hazard de 0,4% nos meses 1–3, 0,9% nos meses 4–9, 0,7% nos meses 10–12, 0,6% depois → F(12) = 8,36%. A régua diz 9,0%. A calibração soma o mesmo δ = 0,077 ao logit de todos os hazards: o formato fica e F(12) passa a 9,0%; a curva chega a 22,1% em 36 meses.

**Kaplan-Meier:** o mesmo produto de sobrevivências, calculado no OOT ou no HIST, é a PD acumulada **observada**. A fórmula de Greenwood dá a variância dela e, com isso, o intervalo de 95%.

**Convenções de idade:**

- idade = meses-calendário desde a safra (mês seguinte à concessão = idade 1);
- default no próprio mês de concessão (idade 0) conta como idade 1;
- contrato censurado na idade 0 (concedido no mês de corte) não entra: não teve exposição.

In [ ]:
%run ./00_config

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

PARTICOES_DES = ("treino", "teste")
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

## 1. Agregação de sobrevivência

Uma linha por rating × safra × partição × duração × evento, com a contagem de contratos. Com 9 ratings, 40 safras e 60 idades, são algumas dezenas de milhares de linhas: tudo o mais (hazard, Kaplan-Meier, mapa de calor) sai dela em pandas.

In [ ]:
def agrega_sobrevivencia(base_com_rating, cfg):
    """
    Contagem de contratos por (rating, safra, particao, duracao, evento).

    duracao = idade no fim da observação (idade do default, se houve).
    evento  = 1 se terminou em default, 0 se foi censurado.
    """
    safra = cfg["colunas"]["safra"]
    duracao = F.greatest(F.col("idade_fim_observacao"), F.lit(0))
    evento = F.col("evento_default")
    duracao = F.when((duracao == 0) & (evento == 1), F.lit(1)).otherwise(duracao)
    return (base_com_rating
            .filter(~((F.col("idade_fim_observacao") <= 0) & (F.col("evento_default") == 0)))
            .groupBy("rating", F.col(safra).alias("safra"), "particao",
                     duracao.alias("duracao"), evento.alias("evento"))
            .count().toPandas())

## 2. Hazard por idade

In [ ]:
def hazard_por_idade(agregado, idade_maxima):
    """
    Tabela por idade t = 1..idade_maxima: em_risco, defaults, hazard (= defaults/em_risco).

    Conceito técnico:
        em_risco(t) = contratos com duração ≥ t  (soma acumulada de trás para frente);
        defaults(t) = contratos com duração = t e evento = 1.
    """
    idades = np.arange(1, idade_maxima + 1)
    # A soma de trás para frente precisa de TODAS as durações, inclusive as maiores
    # que idade_maxima: um contrato acompanhado por 60 meses está em risco na idade 12.
    maior_duracao = int(max(idade_maxima, agregado["duracao"].max() if len(agregado) else 0))
    por_duracao = agregado.groupby("duracao")["count"].sum().reindex(range(0, maior_duracao + 1), fill_value=0)
    eventos = agregado[agregado["evento"] == 1].groupby("duracao")["count"].sum().reindex(idades, fill_value=0)
    em_risco_todas = por_duracao[::-1].cumsum()[::-1]           # contratos com duração ≥ t
    tabela = pd.DataFrame({"idade": idades, "em_risco": em_risco_todas.reindex(idades).to_numpy(),
                           "defaults": eventos.to_numpy()})
    tabela["hazard"] = np.where(tabela["em_risco"] > 0, tabela["defaults"] / tabela["em_risco"].clip(lower=1), np.nan)
    return tabela

## 3. Curva do DES, extrapolação e calibração

**Extrapolação.** Uma idade com menos de `min_em_risco` contratos (100, por padrão) não estima o hazard com confiança: 2 defaults em 40 contratos dão 5%, 0 defaults dão 0%. Essas idades (normalmente as mais altas, que o DES ainda não alcançou) recebem o hazard **agregado das últimas 6 idades confiáveis** (soma dos defaults / soma dos em risco). A coluna `fonte` registra qual idade é observada e qual é extrapolada.

**Calibração.** Para cada rating, resolve δ tal que F(12) = PD da régua:

```
hazard_calibrado(t) = 1 / (1 + e^−(logit(hazard(t)) + δ))
```

Mesmo δ em todas as idades: o formato da maturação (onde está o pico) é o da safra; o nível é o da régua.

In [ ]:
def curva_do_rating(agregado_rating, cfg, idade_maxima):
    """Hazard bruto (DES), extrapolação e colunas de sobrevivência, para um rating."""
    l = cfg["lifetime"]
    des = agregado_rating[agregado_rating["particao"].isin(PARTICOES_DES)]
    curva = hazard_por_idade(des, idade_maxima)
    confiavel = curva["em_risco"] >= l["min_em_risco"]
    curva["fonte"] = np.where(confiavel, "observada", "extrapolada")
    if not confiavel.any():
        raise ValueError("Nenhuma idade com contratos suficientes em risco para estimar o hazard.")
    hazard = curva["hazard"].copy()
    for posicao in np.where(~confiavel)[0]:
        anteriores = curva.loc[:posicao][confiavel.loc[:posicao]].tail(l["n_idades_extrapolacao"])
        if anteriores.empty:
            anteriores = curva[confiavel].head(l["n_idades_extrapolacao"])
        hazard.iloc[posicao] = anteriores["defaults"].sum() / anteriores["em_risco"].sum()
    curva["hazard_bruto"] = hazard.clip(1e-8, 1 - 1e-8)
    curva["pd_acumulada_bruta"] = 1 - np.cumprod(1 - curva["hazard_bruto"])
    return curva


def calibra_curva(curva, pd_alvo_12m):
    """Acrescenta hazard_calibrado, sobrevivencia, pd_acumulada e pd_marginal (δ no logit)."""
    from scipy.optimize import brentq
    logit = np.log(curva["hazard_bruto"] / (1 - curva["hazard_bruto"]))

    def pd_12m(delta):
        h = 1 / (1 + np.exp(-(logit[:12] + delta)))
        return 1 - np.prod(1 - h)

    delta = 0.0 if not (0 < pd_alvo_12m < 1) else brentq(lambda d: pd_12m(d) - pd_alvo_12m, -20, 20)
    curva = curva.copy()
    curva["delta_calibracao"] = delta
    curva["hazard_calibrado"] = 1 / (1 + np.exp(-(logit + delta)))
    curva["sobrevivencia"] = np.cumprod(1 - curva["hazard_calibrado"])
    curva["pd_acumulada"] = 1 - curva["sobrevivencia"]
    sobrevivencia_anterior = np.concatenate([[1.0], curva["sobrevivencia"].to_numpy()[:-1]])
    curva["pd_marginal"] = sobrevivencia_anterior * curva["hazard_calibrado"]
    return curva

## 4. Backtest com Kaplan-Meier e Greenwood

Para cada rating e cada fonte (OOT e HIST), a PD acumulada observada por Kaplan-Meier:

```
S_KM(t) = Π (1 − d_k / n_k)              d = defaults, n = em risco, k = 1..t
Var[S_KM(t)] = S_KM(t)² · Σ d_k / (n_k · (n_k − d_k))      (Greenwood)
IC 95% de F(t) = 1 − S_KM(t) ± 1,96 · √Var
```

Em cada horizonte (12, 24, 36 meses), a linha registra a PD prevista (curva calibrada), a observada, o intervalo, quantos contratos estavam em risco naquela idade e o veredito. Se houver menos de `min_em_risco` contratos em risco no horizonte, o veredito é **sem dados**: o OOT, por ser recente, normalmente só testa 12 meses, e o HIST testa os horizontes longos (decisão do desenho).

In [ ]:
def kaplan_meier(agregado, idade_maxima, nivel=0.95):
    """PD acumulada observada por Kaplan-Meier, com intervalo de Greenwood."""
    from scipy.stats import norm
    t = hazard_por_idade(agregado, idade_maxima)
    n, d = t["em_risco"].to_numpy(float), t["defaults"].to_numpy(float)
    razao = np.where(n > 0, d / np.maximum(n, 1), 0.0)
    sobrevivencia = np.cumprod(1 - razao)
    termo = np.where((n > 0) & (n > d), d / (np.maximum(n, 1) * np.maximum(n - d, 1)), 0.0)
    erro_padrao = sobrevivencia * np.sqrt(np.cumsum(termo))
    z = norm.ppf(1 - (1 - nivel) / 2)
    t["pd_observada"] = 1 - sobrevivencia
    t["ic_inf"] = np.clip(t["pd_observada"] - z * erro_padrao, 0, 1)
    t["ic_sup"] = np.clip(t["pd_observada"] + z * erro_padrao, 0, 1)
    return t


def backtest_lifetime(agregado, curvas, cfg):
    """Uma linha por rating × fonte × horizonte."""
    l = cfg["lifetime"]
    idade_maxima = max(l["horizontes_backtest"])
    linhas = []
    for rating, curva in curvas.items():
        for fonte in ("oot", "hist"):
            dados = agregado[(agregado["rating"] == rating) & (agregado["particao"] == fonte)]
            if dados.empty:
                continue
            km = kaplan_meier(dados, idade_maxima).set_index("idade")
            for horizonte in l["horizontes_backtest"]:
                prevista = float(curva.set_index("idade").loc[horizonte, "pd_acumulada"])
                observada, em_risco = km.loc[horizonte, ["pd_observada", "em_risco"]]
                tem_dados = em_risco >= l["min_em_risco"]
                dentro = bool(km.loc[horizonte, "ic_inf"] <= prevista <= km.loc[horizonte, "ic_sup"])
                linhas.append({"rating": rating, "fonte": fonte, "horizonte_meses": horizonte,
                               "pd_prevista": prevista, "pd_observada": float(observada),
                               "ic_inf": float(km.loc[horizonte, "ic_inf"]), "ic_sup": float(km.loc[horizonte, "ic_sup"]),
                               "em_risco_no_horizonte": int(em_risco),
                               "diferenca": float(observada - prevista),
                               "veredito": ("sem dados" if not tem_dados
                                            else "dentro do IC" if dentro
                                            else "acima do IC" if observada > prevista else "abaixo do IC")})
    return pd.DataFrame(linhas)

## 5. Aplicação aos contratos vivos

Contratos **vivos na data de corte**: sem default e com o fim da observação igual à data de corte. Para um contrato do rating r com idade atual a:

```
PD 12 meses (vista de hoje) = 1 − S(a + 12) / S(a)
PD lifetime                 = 1 − S(a + horizonte) / S(a)       (horizonte = 60 meses)
```

A curva começa na idade atual do contrato, não no mês 1. Exemplo: rating H, idade 6 → as próximas 12 marginais são as das idades 7 a 18, depois do pico de risco das idades 4 a 9; a PD de 12 meses do contrato (8,7%) fica abaixo da do rating (9,0%).

Sem a coluna de prazo (decisão do desenho: só colunas essenciais), a PD lifetime é calculada para o horizonte fixo da configuração, e não até o fim de cada contrato.

O cálculo é feito uma vez por (rating, idade) em pandas e levado aos contratos por junção no Spark.

In [ ]:
def pd_por_rating_e_idade(curvas, horizonte, idade_atual_maxima):
    """Tabela (rating, idade_atual) → pd_12m_curva, pd_lifetime."""
    linhas = []
    for rating, curva in curvas.items():
        s = np.concatenate([[1.0], curva["sobrevivencia"].to_numpy()])     # S(0) = 1
        for a in range(0, idade_atual_maxima + 1):
            linhas.append({"rating": rating, "idade_atual": a,
                           "pd_12m_curva": 1 - s[min(a + 12, len(s) - 1)] / s[a],
                           "pd_lifetime": 1 - s[min(a + horizonte, len(s) - 1)] / s[a]})
    return pd.DataFrame(linhas)


def aplica_lifetime(spark, base_com_rating, curvas, cfg, idade_atual_maxima):
    """Contratos vivos na data de corte, com pd_12m_curva e pd_lifetime (Spark)."""
    colunas = cfg["colunas"]
    vivos = (base_com_rating
             .filter((F.col("evento_default") == 0)
                     & (F.col(colunas["data_fim_observacao"]) == F.to_date(F.lit(cfg["data_corte"]))))
             .withColumn("idade_atual", F.least(F.col("meses_ate_corte"), F.lit(idade_atual_maxima))))
    tabela = spark.createDataFrame(pd_por_rating_e_idade(curvas, cfg["lifetime"]["horizonte_meses"],
                                                         idade_atual_maxima))
    return vivos.join(F.broadcast(tabela), on=["rating", "idade_atual"], how="left").select(
        colunas["id"], colunas["safra"], "particao", "rating", "idade_atual", "valor_regua",
        "pd_12m_curva", "pd_lifetime")

## 6. Gráficos 8.1 a 8.5

| gráfico | o que mostra |
|---|---|
| 8.1 mapa safra × idade | PD acumulada observada (Kaplan-Meier) de cada safra em cada idade; a partição de cada safra (HIST, treino, teste, OOT) ao lado. Escurecer na horizontal = maturação; um bloco de safras mais escuro = deterioração da concessão |
| 8.2 hazard por idade | hazard observado do DES com intervalo de 95% (Jeffreys), idades extrapoladas tracejadas e hazard calibrado |
| 8.3 curva bruta × calibrada | PD acumulada antes e depois da calibração, com o ponto de 12 meses na PD da régua; e uma versão com todos os ratings |
| 8.4 prevista × observada | curva calibrada contra Kaplan-Meier do OOT e do HIST, com as bandas de Greenwood |
| 8.5 PD marginal de um contrato | barras mês a mês para um contrato típico do rating (idade atual = mediana dos vivos) |

In [ ]:
def _rotulo_safra(valor):
    data = pd.Timestamp(valor)
    return f"{MESES[data.month - 1]}/{data.year % 100:02d}"


def grafico_mapa_safra_idade(agregado, cfg, rating=None, idade_maxima=36):
    """8.1: PD acumulada observada por safra e idade."""
    import matplotlib.pyplot as plt
    dados = agregado if rating is None else agregado[agregado["rating"] == rating]
    safras = sorted(dados["safra"].unique())
    matriz = np.full((len(safras), idade_maxima), np.nan)
    particoes = []
    for i, safra in enumerate(safras):
        da_safra = dados[dados["safra"] == safra]
        particoes.append(da_safra["particao"].iloc[0])
        km = kaplan_meier(da_safra, idade_maxima)
        validas = km["em_risco"] >= 30
        matriz[i, validas.to_numpy()] = km.loc[validas, "pd_observada"].to_numpy() * 100
    figura, eixo = plt.subplots(figsize=(11, max(3.5, 0.18 * len(safras) + 1.5)), dpi=130)
    imagem = eixo.imshow(matriz, aspect="auto", cmap="Blues", interpolation="nearest")
    eixo.set_yticks(range(len(safras)))
    eixo.set_yticklabels([f"{_rotulo_safra(s)}  {p.upper()}" for s, p in zip(safras, particoes)], fontsize=6)
    eixo.set_xticks(range(0, idade_maxima, 3))
    eixo.set_xticklabels(range(1, idade_maxima + 1, 3), fontsize=7)
    eixo.set_xlabel("idade do contrato (meses)")
    figura.colorbar(imagem, ax=eixo, label="PD acumulada observada (%)")
    eixo.set_title("PD acumulada por safra e idade" + (f" — rating {rating}" if rating else ""),
                   loc="left", fontsize=11)
    figura.tight_layout()
    return figura


def grafico_hazard(curva, rating, cfg, idade_maxima=36):
    """8.2: hazard observado (com IC de Jeffreys), extrapolado e calibrado."""
    import matplotlib.pyplot as plt
    from scipy.stats import beta
    c = curva[curva["idade"] <= idade_maxima]
    observada = c["fonte"] == "observada"
    inferior = beta.ppf(0.025, c["defaults"] + 0.5, c["em_risco"] - c["defaults"] + 0.5)
    superior = beta.ppf(0.975, c["defaults"] + 0.5, c["em_risco"] - c["defaults"] + 0.5)
    figura, eixo = plt.subplots(figsize=(9, 4), dpi=130)
    eixo.fill_between(c["idade"][observada], inferior[observada] * 100, superior[observada] * 100,
                      color="#cde2fb", lw=0, label="IC 95% do hazard observado")
    eixo.plot(c["idade"][observada], c["hazard_bruto"][observada] * 100, color="#2a78d6", marker="o",
              markersize=3, lw=1.5, label="hazard observado (DES)")
    if (~observada).any():
        eixo.plot(c["idade"][~observada], c["hazard_bruto"][~observada] * 100, color="#2a78d6",
                  lw=1.5, ls=(0, (4, 3)), label="extrapolado")
    eixo.plot(c["idade"], c["hazard_calibrado"] * 100, color="#eb6834", lw=2, label="calibrado")
    eixo.set_xlabel("idade do contrato (meses)")
    eixo.set_ylabel("hazard mensal (%)")
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(color="#e4e3df", lw=0.8)
    eixo.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    eixo.set_title(f"Rating {rating}: PD condicional (hazard) por idade", loc="left", fontsize=11)
    figura.tight_layout()
    return figura


def grafico_curva(curva, rating, pd_regua, cfg):
    """8.3: PD acumulada bruta × calibrada."""
    import matplotlib.pyplot as plt
    c = curva[curva["idade"] <= cfg["lifetime"]["horizonte_meses"]]
    figura, eixo = plt.subplots(figsize=(9, 4), dpi=130)
    eixo.plot(c["idade"], c["pd_acumulada_bruta"] * 100, color="#86b6ef", lw=2, label="curva da safra (bruta)")
    eixo.plot(c["idade"], c["pd_acumulada"] * 100, color="#2a78d6", lw=2.2, label="calibrada à régua")
    eixo.scatter([12], [pd_regua * 100], color="#eb6834", zorder=5, s=40, label=f"PD da régua em 12 meses ({pd_regua:.2%})")
    for horizonte in cfg["lifetime"]["horizontes_backtest"]:
        valor = float(c.loc[c["idade"] == horizonte, "pd_acumulada"].iloc[0]) if horizonte <= c["idade"].max() else None
        if valor is not None:
            eixo.annotate(f"{valor:.1%}".replace(".", ","), (horizonte, valor * 100), textcoords="offset points",
                          xytext=(0, 6), ha="center", fontsize=8, color="#52514e")
    eixo.set_xlabel("idade do contrato (meses)")
    eixo.set_ylabel("PD acumulada (%)")
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(color="#e4e3df", lw=0.8)
    eixo.legend(frameon=False, fontsize=8, loc="upper left")
    eixo.set_title(f"Rating {rating}: PD acumulada, bruta × calibrada (δ = {c['delta_calibracao'].iloc[0]:.3f})",
                   loc="left", fontsize=11)
    figura.tight_layout()
    return figura


def grafico_curvas_todos(curvas, cfg):
    """8.3 (todos os ratings): PD acumulada calibrada em escala log."""
    import matplotlib.pyplot as plt
    figura, eixo = plt.subplots(figsize=(9, 4.5), dpi=130)
    cores = plt.cm.Blues(np.linspace(0.35, 1.0, len(curvas)))
    for cor, (rating, curva) in zip(cores, curvas.items()):
        c = curva[curva["idade"] <= cfg["lifetime"]["horizonte_meses"]]
        eixo.plot(c["idade"], c["pd_acumulada"] * 100, color=cor, lw=1.8)
        eixo.text(c["idade"].iloc[-1] + 0.5, c["pd_acumulada"].iloc[-1] * 100, rating, fontsize=8, va="center")
    eixo.set_yscale("log")
    eixo.set_xlabel("idade do contrato (meses)")
    eixo.set_ylabel("PD acumulada (%), escala log")
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(color="#e4e3df", lw=0.8, which="both")
    eixo.set_title("PD acumulada calibrada, todos os ratings", loc="left", fontsize=11)
    figura.tight_layout()
    return figura


def grafico_backtest_curva(agregado, curva, rating, cfg):
    """8.4: curva calibrada × Kaplan-Meier do OOT e do HIST, com bandas de Greenwood."""
    import matplotlib.pyplot as plt
    l = cfg["lifetime"]
    idade_maxima = max(l["horizontes_backtest"])
    c = curva[curva["idade"] <= idade_maxima]
    figura, eixo = plt.subplots(figsize=(9, 4.2), dpi=130)
    eixo.plot(c["idade"], c["pd_acumulada"] * 100, color="#2a78d6", lw=2.2, label="prevista (calibrada)")
    for fonte, cor in (("oot", "#eb6834"), ("hist", "#52514e")):
        dados = agregado[(agregado["rating"] == rating) & (agregado["particao"] == fonte)]
        if dados.empty:
            continue
        km = kaplan_meier(dados, idade_maxima)
        km = km[km["em_risco"] >= l["min_em_risco"]]
        if km.empty:
            continue
        eixo.fill_between(km["idade"], km["ic_inf"] * 100, km["ic_sup"] * 100, color=cor, alpha=0.15, lw=0)
        eixo.plot(km["idade"], km["pd_observada"] * 100, color=cor, lw=1.6, label=f"observada {fonte.upper()} (Kaplan-Meier)")
    for horizonte in l["horizontes_backtest"]:
        eixo.axvline(horizonte, color="#b4b2a9", lw=0.8, ls=(0, (4, 3)))
    eixo.set_xlabel("idade do contrato (meses)")
    eixo.set_ylabel("PD acumulada (%)")
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(color="#e4e3df", lw=0.8)
    eixo.legend(frameon=False, fontsize=8, loc="upper left")
    eixo.set_title(f"Rating {rating}: PD acumulada prevista × observada", loc="left", fontsize=11)
    figura.tight_layout()
    return figura


def grafico_marginal_contrato(curva, rating, idade_atual, cfg):
    """8.5: PD marginal mês a mês de um contrato com a idade atual dada."""
    import matplotlib.pyplot as plt
    horizonte = cfg["lifetime"]["horizonte_meses"]
    s = np.concatenate([[1.0], curva["sobrevivencia"].to_numpy()])
    h = curva["hazard_calibrado"].to_numpy()
    meses = np.arange(1, horizonte + 1)
    idades = idade_atual + meses
    idades = idades[idades <= len(h)]
    marginal = s[idades - 1] / s[idade_atual] * h[idades - 1]
    pd_12m, pd_vida = marginal[:12].sum(), marginal.sum()
    figura, eixo = plt.subplots(figsize=(9, 3.8), dpi=130)
    eixo.bar(meses[:len(marginal)], marginal * 100, color=np.where(meses[:len(marginal)] <= 12, "#2a78d6", "#86b6ef"), width=0.8)
    eixo.set_xlabel(f"meses a partir de hoje (contrato com idade atual {idade_atual})")
    eixo.set_ylabel("PD marginal (%)")
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(axis="y", color="#e4e3df", lw=0.8)
    eixo.set_axisbelow(True)
    eixo.set_title(f"Rating {rating}: PD 12 meses = {pd_12m:.2%} | PD lifetime ({horizonte} meses) = {pd_vida:.2%}"
                   .replace(".", ","), loc="left", fontsize=11)
    figura.tight_layout()
    return figura

## 7. Execução da etapa 8

In [ ]:
def executa_etapa_lifetime(spark, base_com_rating, regua, cfg, gravar=True):
    """
    Etapa 8 (só PD).

    Devolve
        curvas            {rating: pandas por idade}
        tabela_curvas     pandas com todas as curvas
        backtest          pandas, rating × fonte × horizonte
        contratos_vivos   Spark, com pd_12m_curva e pd_lifetime
        resumo_vivos      pandas por rating
        figuras           {nome: figura ou {rating: figura}}
    """
    if cfg["tarefa"] != "pd":
        raise ValueError("A PD lifetime só se aplica à tarefa PD.")
    l = cfg["lifetime"]
    agregado = agrega_sobrevivencia(base_com_rating, cfg)
    idade_atual_maxima = int(base_com_rating.agg(F.max("meses_ate_corte")).first()[0])
    idade_maxima = max(l["horizonte_meses"] + idade_atual_maxima, max(l["horizontes_backtest"]))
    pd_regua = regua.set_index("rating")["valor_regua"].to_dict()

    curvas = {}
    for rating in regua["rating"]:
        bruta = curva_do_rating(agregado[agregado["rating"] == rating], cfg, idade_maxima)
        curvas[rating] = calibra_curva(bruta, pd_regua[rating])
    tabela_curvas = pd.concat([c.assign(rating=r) for r, c in curvas.items()], ignore_index=True)
    resumo_curvas = pd.DataFrame([{
        "rating": r, "pd_regua_12m": pd_regua[r],
        "pd_12m_bruta": float(c.loc[c["idade"] == 12, "pd_acumulada_bruta"].iloc[0]),
        "delta_calibracao": float(c["delta_calibracao"].iloc[0]),
        **{f"pd_{h}m": float(c.loc[c["idade"] == h, "pd_acumulada"].iloc[0]) for h in l["horizontes_backtest"]},
        "pd_lifetime_horizonte": float(c.loc[c["idade"] == l["horizonte_meses"], "pd_acumulada"].iloc[0]),
        "ultima_idade_observada": int(c.loc[c["fonte"] == "observada", "idade"].max()),
    } for r, c in curvas.items()])
    exibe(resumo_curvas, "Curvas por rating (calibradas à régua)")

    backtest = backtest_lifetime(agregado, curvas, cfg)
    exibe(backtest, "Backtest da PD lifetime")

    contratos_vivos = aplica_lifetime(spark, base_com_rating, curvas, cfg, idade_atual_maxima).persist()
    resumo_vivos = (contratos_vivos.groupBy("rating")
                    .agg(F.count(F.lit(1)).alias("contratos_vivos"), F.avg("idade_atual").alias("idade_media"),
                         F.expr("percentile_approx(idade_atual, 0.5)").alias("idade_mediana"),
                         F.avg("valor_regua").alias("pd_regua"), F.avg("pd_12m_curva").alias("pd_12m_media"),
                         F.avg("pd_lifetime").alias("pd_lifetime_media"))
                    .orderBy("rating").toPandas())
    exibe(resumo_vivos, "Contratos vivos na data de corte")

    medianas = resumo_vivos.set_index("rating")["idade_mediana"].to_dict()
    figuras = {
        "mapa_safra_idade": grafico_mapa_safra_idade(agregado, cfg),
        "mapa_safra_idade_por_rating": {r: grafico_mapa_safra_idade(agregado, cfg, r) for r in curvas},
        "hazard": {r: grafico_hazard(c, r, cfg) for r, c in curvas.items()},
        "curva": {r: grafico_curva(c, r, pd_regua[r], cfg) for r, c in curvas.items()},
        "curvas_todos": grafico_curvas_todos(curvas, cfg),
        "backtest_curva": {r: grafico_backtest_curva(agregado, c, r, cfg) for r, c in curvas.items()},
        "marginal_contrato": {r: grafico_marginal_contrato(c, r, int(medianas.get(r) or 6), cfg)
                              for r, c in curvas.items()},
    }
    for nome, item in figuras.items():
        if isinstance(item, dict):
            for rating, figura in item.items():
                registra_figura_mlflow(figura, f"lifetime/{nome}_{rating}.png")
        else:
            registra_figura_mlflow(item, f"lifetime/{nome}.png")

    if gravar:
        grava_tabela_delta(spark, tabela_curvas, "lifetime_curvas", cfg)
        grava_tabela_delta(spark, resumo_curvas, "lifetime_resumo", cfg)
        grava_tabela_delta(spark, backtest, "lifetime_backtest", cfg)
        grava_tabela_delta(spark, resumo_vivos, "lifetime_vivos_resumo", cfg)

    return {"curvas": curvas, "tabela_curvas": tabela_curvas, "resumo_curvas": resumo_curvas,
            "backtest": backtest, "contratos_vivos": contratos_vivos, "resumo_vivos": resumo_vivos,
            "figuras": figuras}

## 8. Exemplo de uso

```python
etapa8 = executa_etapa_lifetime(spark, etapa7["base_com_rating"], etapa7["regua"], cfg)
etapa8["resumo_curvas"]                      # PD 12, 24, 36 meses e lifetime por rating
etapa8["backtest"]                           # prevista × observada (OOT e HIST)
etapa8["figuras"]["backtest_curva"]["H"]     # gráfico 8.4 do rating H
```